<a href="https://colab.research.google.com/github/k8peevey-sys/ds2002-fa26/blob/main/Copy_of_2026_10_02_%E2%80%94_Normalize_and_Join_Nested_JSON_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-01','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** _..._

1.   List item
2.   List item



### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [2]:
import pandas as pd
#Here I am flattening the nested list and creating rows from items inside the lines and then copying them onto every row
lines = pd.json_normalize(orders_json, record_path='lines',meta=['order', 'vendor_id'])
print (lines)

assert len(lines) == 4
assert lines['qty'].sum() == 7

           item  qty order vendor_id
0  Cheeseburger    2     1      V-01
1          Soda    1     1      V-01
2   Foam Finger    1     2      V-10
3       Hot Dog    3     3      V-01


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [5]:
vendors = pd.DataFrame(vendors_json)
#Here I am left joining
joined = lines.merge(vendors, on='vendor_id', how='left', indicator=True)
print(joined)
#Here I am creating an index that will report any unmatched vendors
unmatched = joined[joined['_merge'] == 'left_only']
print('Unmatched:',unmatched)

           item  qty order vendor_id          name zone _merge
0  Cheeseburger    2     1      V-01  Hoos Burgers    A   both
1          Soda    1     1      V-01  Hoos Burgers    A   both
2   Foam Finger    1     2      V-10     Cav Merch    A   both
3       Hot Dog    3     3      V-01  Hoos Burgers    A   both
Unmatched: Empty DataFrame
Columns: [item, qty, order, vendor_id, name, zone, _merge]
Index: []


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [7]:
#Here I am grouping by vendor name in order to count the total quantity by vendor
#By using dropna= False, if an unmatched vendor appears it will show up as NaN instad of disappearing
vendor_totals = (joined.groupby('name', dropna=False)['qty'].sum().reset_index() )
print(vendor_totals)

           name  qty
0     Cav Merch    1
1  Hoos Burgers    6


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [9]:
# We know that every vendor belongs to a zone so I am just grouping by zone
zone_totals = (joined.groupby('zone', dropna=False)['qty'].sum().reset_index() )
print(zone_totals)

#Here I am calculating the largest zone
top_zone = zone_totals.loc[zone_totals['qty'].idxmax()]
print('Top zone:', top_zone)

  zone  qty
0    A    7
Top zone: zone    A
qty     7
Name: 0, dtype: object


Zone A sold the most items with 7 units


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [10]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

records = []
for order in ragged_json:
    for line in order.get('lines', []):
        records.append({
            'order': order['order'],
            'vendor_id': order['vendor_id'],
            'item': line['item'],
            'qty': line.get('qty')})
ragged_lines = pd.DataFrame(records)
print(ragged_lines)

   order vendor_id   item  qty
0      4      V-01   Soda  2.0
1      6      V-01  Fries  NaN


### Q6 — Validate

In [11]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?



```
# This is formatted as code
```

_a) One row per  line is useful because each rows is only one item being sold, so having one row per line makes summation and analysis easy.
b) A missing quantity is unknown while a value of 0 means that the quantity actually is zero. Replacing missing quantities with 0 leads to averages that are incorrect because there are too many zero values. _